### Code chạy 4 model Lgbm + XGB + RF + SVR

In [ ]:
import os
import rasterio
import numpy as np
import joblib
from tqdm import tqdm

# ===============================
# 1. PATHS - CẬP NHẬT NẾU CẦN
# ===============================
SEASONAL_DIR = r"E:\DownloadData\co2_ban_do\output_region_seasonal"
SOIL_DIR     = r"E:\DownloadData\co2_ban_do\output_region_soilgrids"   # nơi bạn lưu soc_0_5_RRD.tif...
RICE_MASK_RRD = r"E:\DownloadData\co2_ban_do\ouput_rice_mask_region\rice_mask_RRD.tif"
RICE_MASK_MKD = r"E:\DownloadData\co2_ban_do\ouput_rice_mask_region\rice_mask_MKD.tif"

MODEL_PATH = r"E:\DownloadData\modelML\final_model\lgbm_model.pkl"
OUTPUT_DIR = r"E:\DownloadData\co2_ban_do\co2_map_new\map_lgbm"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===============================
# 2. SOIL + SEASONAL feature order
# (soil BEFORE seasonal to match model if trained that way)
# ===============================
SOIL_FEATURES = [
    "soc_0_5", "soc_5_15", "soc_15_30",
    "ph_0_5",  "ph_5_15",  "ph_15_30"
]

SEASONAL_FEATURES = [
    "precipitation",
    "temperature_2m", "skin_temperature", "soil_temperature_L1",
    "soil_water_L1", "surface_solar_radiation", "total_precipitation",
    "lai_low_veg", "wind_u_10m", "wind_v_10m",
    "dewpoint_temp_2m", "surface_pressure",
    "ET", "LE", "PET", "PLE",
    "FPAR", "LAI",
    "NDVI", "EVI", "LSWI", "NDWI_McFeeters",
    "AOD_047um", "AOD_055um",
    "GMT_0000_PAR", "GMT_0300_PAR", "GMT_0600_PAR", "GMT_0900_PAR",
    "sm_surface_daily"
]

# Final order used to build X (soil first, then seasonal)
FEATURE_ORDER = SOIL_FEATURES + SEASONAL_FEATURES
print("Total features expected:", len(FEATURE_ORDER))

# ===============================
# 3. LOAD MODEL
# ===============================
model = joblib.load(MODEL_PATH)
print("✔ Loaded model:", MODEL_PATH)

# ===============================
# 4. LOAD RICE MASK PER REGION
# ===============================
region_rice_mask = {}
region_mask_meta = {}

for region, path in {"RRD": RICE_MASK_RRD, "MKD": RICE_MASK_MKD}.items():
    with rasterio.open(path) as src:
        arr = src.read(1)
        meta = src.meta.copy()
    # assume rice mask uses 1 for rice (or True), adjust accordingly
    region_rice_mask[region] = (arr == 1)
    region_mask_meta[region] = meta
    print(f"✔ Loaded rice mask for {region}: {path}, pixels={np.sum(region_rice_mask[region])}")

# ===============================
# 5. REGION / SEASON / YEARS
# ===============================
regions = ["RRD", "MKD"]
SEASONS = {"RRD": ["Xuan", "Mua", "Dong"], "MKD": ["DongXuan", "HeThu", "ThuDong"]}
YEARS = [2020,2021]   # your requested years

# ===============================
# 6. FUNCTIONS: load soil stack & seasonal stack
# ===============================
def load_soil_stack(region):
    """Load all soil bands for a region and return (n_soil, H, W)."""
    bands = []
    for var in SOIL_FEATURES:
        fp = os.path.join(SOIL_DIR, f"{var}_{region}.tif")
        if not os.path.exists(fp):
            raise FileNotFoundError(f"Missing soil file: {fp}")
        with rasterio.open(fp) as src:
            arr = src.read(1).astype("float32")
        bands.append(arr)
    stack = np.stack(bands, axis=0)
    return stack

def load_seasonal_stack(region, year, season):
    """Load seasonal features (each feature is a single-band tif saved under SEASONAL_DIR/var/region/var_region_year_season.tif).
       Return (stack, meta) where stack shape = (n_features, H, W).
    """
    bands = []
    meta = None
    for var in SEASONAL_FEATURES:
        tif_path = os.path.join(SEASONAL_DIR, var, region, f"{var}_{region}_{year}_{season}.tif")
        if not os.path.exists(tif_path):
            print("❌ Missing seasonal file:", tif_path)
            return None, None
        with rasterio.open(tif_path) as src:
            arr = src.read(1).astype("float32")
            if meta is None:
                meta = src.meta.copy()
        bands.append(arr)
    stack = np.stack(bands, axis=0)
    return stack, meta

# ===============================
# 7. MAIN LOOP: load soil per region once, then seasonal per year/season
# ===============================
for region in regions:
    print("\n====================================")
    print("PROCESS REGION:", region)
    print("====================================")

    # load soil stack for this region (static)
    try:
        soil_stack = load_soil_stack(region)   # shape (n_soil, H, W)
    except FileNotFoundError as e:
        print("ERROR loading soil:", e)
        print("Skip region:", region)
        continue

    # get rice mask and meta
    rice_mask = region_rice_mask[region]
    meta_ref = region_mask_meta[region]

    for year in YEARS:
        for season in SEASONS[region]:
            print(f"\n→ {region} - {year} - {season}")

            seasonal_stack, meta = load_seasonal_stack(region, year, season)
            if seasonal_stack is None:
                print("⚠ Skip due to missing seasonal data.")
                continue

            # Concatenate soil + seasonal along band axis
            # shapes must match on H,W
            if soil_stack.shape[1:] != seasonal_stack.shape[1:]:
                print("❌ SIZE MISMATCH between soil and seasonal stacks!")
                print("soil:", soil_stack.shape, "seasonal:", seasonal_stack.shape)
                print("You need to align grids; skipping.")
                continue

            feature_stack = np.concatenate([soil_stack, seasonal_stack], axis=0)  # (n_total, H, W)

            # Check shape vs rice_mask
            if feature_stack.shape[1:] != rice_mask.shape:
                print("❌ SIZE MISMATCH with rice_mask:", feature_stack.shape, rice_mask.shape)
                print("Skipping.")
                continue

            # Extract raster values at rice pixels only
            X = feature_stack[:, rice_mask].T   # (n_pixels, n_features)
            print("✔ Predicting on shape:", X.shape)

            # Predict
            try:
                y = model.predict(X)
            except Exception as e:
                print("Model predict error:", e)
                continue

            # write output map (only rice pixels filled)
            out_map = np.full(rice_mask.shape, np.nan, dtype="float32")
            out_map[rice_mask] = y.astype("float32")

            # prepare metadata for saving (use seasonal meta if available, else rice meta)
            out_meta = meta.copy() if meta is not None else meta_ref.copy()
            out_meta.update({"count": 1, "dtype": "float32", "compress": "lzw"})

            out_name = f"CO2_{region}_{year}_{season}.tif"
            out_path = os.path.join(OUTPUT_DIR, out_name)
            with rasterio.open(out_path, "w", **out_meta) as dst:
                dst.write(out_map, 1)

            print("✔ Saved:", out_path)

print("\n🎉 ALL DONE.")


### code chạy stacking

In [ ]:
import os
import rasterio
import numpy as np
import cloudpickle
from tqdm import tqdm
from sklearn.impute import SimpleImputer

# ===============================
# 1. PATHS
# ===============================
SEASONAL_DIR = r"E:\DownloadData\co2_ban_do\output_region_seasonal"
SOIL_DIR     = r"E:\DownloadData\co2_ban_do\output_region_soilgrids"
RICE_MASK_RRD = r"E:\DownloadData\co2_ban_do\ouput_rice_mask_region\rice_mask_RRD.tif"
RICE_MASK_MKD = r"E:\DownloadData\co2_ban_do\ouput_rice_mask_region\rice_mask_MKD.tif"

MODEL_PATH = r"E:\DownloadData\modelML\final_model\stack_model.pkl"
OUTPUT_DIR = r"E:\DownloadData\co2_ban_do\co2_map_new\map_stacking"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===============================
# 2. FEATURE ORDER
# ===============================
SOIL_FEATURES = [
    "soc_0_5", "soc_5_15", "soc_15_30",
    "ph_0_5",  "ph_5_15",  "ph_15_30"
]

SEASONAL_FEATURES = [
    "precipitation",
    "temperature_2m", "skin_temperature", "soil_temperature_L1",
    "soil_water_L1", "surface_solar_radiation", "total_precipitation",
    "lai_low_veg", "wind_u_10m", "wind_v_10m",
    "dewpoint_temp_2m", "surface_pressure",
    "ET", "LE", "PET", "PLE",
    "FPAR", "LAI",
    "NDVI", "EVI", "LSWI", "NDWI_McFeeters",
    "AOD_047um", "AOD_055um",
    "GMT_0000_PAR", "GMT_0300_PAR", "GMT_0600_PAR", "GMT_0900_PAR",
    "sm_surface_daily"
]

FEATURE_ORDER = SOIL_FEATURES + SEASONAL_FEATURES
print("Total features expected:", len(FEATURE_ORDER))

# ===============================
# 3. LOAD STACKING MODEL
# ===============================
with open(MODEL_PATH, "rb") as f:
    model = cloudpickle.load(f)
print("✔ Loaded STACKING model:", MODEL_PATH)

# ===============================
# 4. LOAD MASK
# ===============================
region_rice_mask = {}
region_mask_meta = {}

for region, path in {"RRD": RICE_MASK_RRD, "MKD": RICE_MASK_MKD}.items():
    with rasterio.open(path) as src:
        arr = src.read(1)
        meta = src.meta.copy()
    region_rice_mask[region] = (arr == 1)
    region_mask_meta[region] = meta
    print(f"✔ Loaded rice mask for {region}: pixels={np.sum(arr==1)}")

# ===============================
# 5. LOAD YEARS & SEASONS
# ===============================
regions = ["RRD", "MKD"]
SEASONS = {
    "RRD": ["Xuan", "Mua", "Dong"],
    "MKD": ["DongXuan", "HeThu", "ThuDong"]
}
YEARS = [2020,2021,2022,2023,2024]

# ===============================
# 6. FUNCTIONS
# ===============================
def load_soil_stack(region):
    bands = []
    for var in SOIL_FEATURES:
        fp = os.path.join(SOIL_DIR, f"{var}_{region}.tif")
        if not os.path.exists(fp):
            raise FileNotFoundError(f"Missing soil file: {fp}")
        with rasterio.open(fp) as src:
            arr = src.read(1).astype("float32")
        bands.append(arr)
    return np.stack(bands, axis=0)


def load_seasonal_stack(region, year, season):
    bands = []
    meta = None
    for var in SEASONAL_FEATURES:
        fp = os.path.join(SEASONAL_DIR, var, region, f"{var}_{region}_{year}_{season}.tif")
        if not os.path.exists(fp):
            print("⚠ Missing:", fp)
            return None, None
        with rasterio.open(fp) as src:
            arr = src.read(1).astype("float32")
            if meta is None:
                meta = src.meta.copy()
        bands.append(arr)
    return np.stack(bands, axis=0), meta


def prepare_features(feature_stack, rice_mask):
    X = feature_stack[:, rice_mask].T
    return X.astype("float32")

# ===============================
# 7. MAIN PROCESS LOOP
# ===============================
for region in regions:
    print("\n==============================")
    print("PROCESS REGION:", region)
    print("==============================")

    try:
        soil_stack = load_soil_stack(region)
    except Exception as e:
        print("❌ Soil error:", e)
        continue

    rice_mask = region_rice_mask[region]
    meta_ref  = region_mask_meta[region]

    for year in YEARS:
        for season in SEASONS[region]:
            print(f"\n>>> {region} - {year} - {season}")

            seasonal_stack, meta = load_seasonal_stack(region, year, season)
            if seasonal_stack is None:
                print("⚠ Skip due to missing seasonal data.")
                continue

            if soil_stack.shape[1:] != seasonal_stack.shape[1:]:
                print("❌ Size mismatch soil vs seasonal")
                continue

            feature_stack = np.concatenate([soil_stack, seasonal_stack], axis=0)

            if feature_stack.shape[1:] != rice_mask.shape:
                print("❌ Size mismatch mask vs features")
                continue

            X = prepare_features(feature_stack, rice_mask)

            # ===============================
            # FIX SVR ERROR: IMPUTE ANY NaN
            # ===============================
            imputer = SimpleImputer(strategy="mean")
            X = imputer.fit_transform(X)

            print("✔ Predicting:", X.shape)

            try:
                y = model.predict(X)
            except Exception as e:
                print("❌ Predict error:", e)
                continue

            out_map = np.full(rice_mask.shape, np.nan, dtype="float32")
            out_map[rice_mask] = y.astype("float32")

            out_meta = (meta.copy() if meta else meta_ref.copy())
            out_meta.update({"count": 1, "dtype": "float32", "compress": "lzw"})

            out_path = os.path.join(OUTPUT_DIR, f"CO2_{region}_{year}_{season}.tif")

            with rasterio.open(out_path, "w", **out_meta) as dst:
                dst.write(out_map, 1)

            print("✔ Saved:", out_path)

print("\n🎉 DONE — CO₂ maps generated successfully via STACKING!")
